# C4 Conformant Development Rerun — T4 GPU

This notebook runs the full C4 conformant development rerun on a T4 GPU.

**What it does:**
- Clones the repository
- Installs dependencies (transformers >= 5.9, torch, etc.)
- Runs the CPU-only dry-run validation (7 conformance gates)
- Runs the C4-BRIDGE gate (no HRM, ~2 seconds)
- Runs the full HRM development run (120 tasks × 7 arms = 840 generations)
- Runs the analyzer and composition diagnostic
- Downloads all results

**Expected time on T4:** ~15-25 minutes (vs ~3+ hours on CPU)

**Key optimizations:**
- HRM model loaded on GPU (fp16)
- BGE embeddings on GPU
- SDPA attention implementation
- Resumability: if Colab disconnects, re-run picks up where it left off


## 0. Setup — Select GPU Runtime

Make sure you have **T4 GPU** enabled:
- Runtime → Change runtime type → Hardware accelerator → T4 GPU
- Runtime shape → High-RAM (recommended)

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("Stopping here. Please restart runtime after enabling GPU.")
    import sys; sys.exit(0)

## 1. Clone Repository

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/dawsonblock/Daph-ex-research-gate-c2-beir-retrieval.git"
REPO_DIR = "/content/Daph-ex-research-gate-c2-beir-retrieval"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}")
    # Pull latest if already cloned
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--rebase"], check=False)
else:
    print(f"Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Git commit: {subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()[:12]}")

## 2. Install Dependencies

We need transformers >= 5.9.0 for the native HRM-Text-1B architecture.

In [ ]:
# Install dependencies
# transformers 5.9+ is required for HrmTextForCausalLM
# On Colab, torch is pre-installed but we ensure a compatible version
print("Installing dependencies...")
subprocess.run(["pip", "install", "-q", "transformers>=5.9.0", "huggingface-hub>=0.34"], check=True)
subprocess.run(["pip", "install", "-q", "rank-bm25", "numpy"], check=True)
subprocess.run(["pip", "install", "-q", "pytest"], check=False)

# Install the package in development mode
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

import transformers
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print("Dependencies installed.")

## 3. Run Tests (Quick Verification)

Verify the repository is in a GREEN state before running.

In [ ]:
# Run the test suite (should be 607 passed, 2 skipped)
print("Running test suite...")
result = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--tb=no"], 
                       capture_output=True, text=True, timeout=120)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("\nWARNING: Some tests failed. Check output above.")
    print(result.stderr[-500:])
else:
    print("\nAll tests passed!")

## 4. CPU-Only Dry Run (Conformance Validation)

This validates all 7 conformance gates before HRM:
1. No oracle leakage
2. Arm parity
3. Selected IDs in pool
4. Packet budgets
5. Q3 query formulation
6. Merge provenance
7. Causal parity

In [ ]:
print("Running CPU-only dry run (conformance validation)...")
print("This takes ~15 seconds on T4 (BGE embeddings on GPU)...")
result = subprocess.run(["python", "scripts/run_gate_c4.py", "dry-run"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if "ALL VALIDATION GATES PASSED" in result.stdout:
    print("\n=== ALL CONFORMANCE GATES PASSED — safe to proceed to HRM ===")
else:
    print("\nERROR: Conformance validation failed!")
    print(result.stderr[-1000:])
    raise RuntimeError("Conformance validation failed")

## 5. C4-BRIDGE Gate (No HRM, ~2 seconds)

This is the bridge qualification gate. It should reproduce the negative result
(no runtime bridge mechanism beats the one-pass baseline).

In [ ]:
print("Running C4-BRIDGE gate (no HRM)...")
result = subprocess.run(["python", "scripts/run_gate_c4_bridge.py"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if "BeatsB0=False" in result.stdout:
    print("\n=== C4-BRIDGE NEGATIVE RESULT confirmed ===")
else:
    print("\nWARNING: Unexpected C4-BRIDGE result")

## 6. HRM Smoke Test (3 tasks × 7 arms)

Quick end-to-end test with HRM to verify the model loads on GPU.

In [ ]:
print("Running HRM smoke test (3 tasks × 7 arms)...")
print("This takes ~2-3 minutes on T4...")
result = subprocess.run(["python", "scripts/run_gate_c4.py", "smoke"],
                       capture_output=True, text=True, timeout=600)
print(result.stdout)
if "smoke test complete" in result.stdout:
    print("\n=== Smoke test PASSED ===")
else:
    print("\nERROR: Smoke test failed!")
    print(result.stderr[-1000:])
    raise RuntimeError("Smoke test failed")

## 7. Full Conformant Development Run (120 tasks × 7 arms)

This is the main run. 840 HRM generations total.

**On T4 GPU: ~15-25 minutes** (vs 3+ hours on CPU)

The run is resumable — if Colab disconnects, just re-run this cell
and it will pick up where it left off.

In [ ]:
# Patch the HRM loader to use GPU with fp16 for maximum speed
# We do this by setting environment variables before running
import os
os.environ["HRM_DEVICE"] = "cuda"
os.environ["HRM_DTYPE"] = "float16"

# Use C4 protocol v2 (deterministic, reproducible)
os.environ["C4_PROTOCOL"] = "v2"

# Check for existing results (resumability)
out_dir = os.path.join(REPO_DIR, "evidence/gate_c4/full/development")
if os.path.exists(out_dir):
    for arm in ["C4_0", "C4_1", "C4_2", "C4_3", "C4_4", "C4_5", "C4_6"]:
        fpath = os.path.join(out_dir, f"{arm}.jsonl")
        if os.path.exists(fpath):
            with open(fpath) as f:
                lines = sum(1 for _ in f if f.strip())
            print(f"  {arm}: {lines}/120 existing results")

print("\n=== Starting Full Conformant Development Run (C4 Protocol v2) ===")
print("120 tasks × 7 arms = 840 HRM generations")
print("Expected time on T4: ~15-25 minutes")
print()

# Run with real-time output streaming
import subprocess
import sys

proc = subprocess.Popen(
    ["python", "scripts/run_gate_c4.py", "full", "--split", "development"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode != 0:
    print(f"\nERROR: Full run failed with exit code {proc.returncode}")
    raise RuntimeError("Full run failed")
else:
    print("\n=== Full Conformant Development Run COMPLETE ===")

## 8. Run Analyzer

Compute all metrics: arm quality, paired deltas, family/cluster/template CIs,
task flips, per-regime breakdown, identity stats, selector stats, CSR,
role retention, gap capture.

In [ ]:
print("Running C4 analyzer...")
result = subprocess.run(
    ["python", "scripts/analyze_gate_c4.py", 
     "--dir", "evidence/gate_c4/full/development",
     "--output", "evidence/gate_c4/full/development/analysis.json"],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.returncode == 0:
    print("\n=== Analysis complete ===")
    print(f"Full report: evidence/gate_c4/full/development/analysis.json")
else:
    print(f"\nAnalyzer error: {result.stderr[-500:]}")

## 9. Run Composition Diagnostic

Diagnose S2c selection behavior and the canonical regression.

In [ ]:
print("Running composition diagnostic...")
result = subprocess.run(
    ["python", "scripts/diagnose_c4_composition.py"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout)

## 10. Verify Results Integrity

Check RESULTS.sha256 and manifest.

In [ ]:
import json
from pathlib import Path

out_dir = Path("evidence/gate_c4/full/development")

# Check manifest
manifest_path = out_dir / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("=== Manifest ===")
    for key in ["mode", "split", "arm_ids", "task_count", "git_commit",
                "protocol_sha256", "hrm_model_id", "hrm_model_revision",
                "hrm_max_new_tokens", "device", "created_utc"]:
        val = manifest.get(key, "N/A")
        if isinstance(val, str) and len(val) > 20:
            val = val[:20] + "..."
        print(f"  {key}: {val}")
else:
    print("WARNING: No manifest found!")

# Check RESULTS.sha256
results_hash_path = out_dir / "RESULTS.sha256"
if results_hash_path.exists():
    print(f"\n=== RESULTS.sha256 ===")
    print(results_hash_path.read_text())
else:
    print("\nWARNING: No RESULTS.sha256 found!")

# Verify per-arm results
print("\n=== Per-Arm Results ===")
for arm_id in ["C4_0", "C4_1", "C4_2", "C4_3", "C4_4", "C4_5", "C4_6"]:
    arm_path = out_dir / f"{arm_id}.jsonl"
    if arm_path.exists():
        lines = [l for l in arm_path.read_text().splitlines() if l.strip()]
        print(f"  {arm_id}: {len(lines)} receipts")
    else:
        print(f"  {arm_id}: MISSING")

## 11. Download Results

Package all results for download.

In [ ]:
import shutil
from google.colab import files

# Create a zip of all C4 evidence
zip_path = "/content/c4_conformant_results.zip"
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', 
                    'evidence/gate_c4')
print(f"Created {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

# Also copy the analysis report
analysis_path = "evidence/gate_c4/full/development/analysis.json"
if os.path.exists(analysis_path):
    shutil.copy(analysis_path, "/content/c4_analysis.json")
    print("Copied analysis to /content/c4_analysis.json")

# Download
files.download(zip_path)

## 12. Summary

### What was run:
1. **Conformance validation** (7 gates, all PASS)
2. **C4-BRIDGE gate** (negative result confirmed — one-pass pipeline)
3. **HRM smoke test** (3 tasks × 7 arms, all passed)
4. **Full conformant development run** (120 tasks × 7 arms = 840 HRM generations)
5. **Analyzer** (quality, deltas, CIs, flips, per-regime, identity, selector, CSR, gap capture)
6. **Composition diagnostic** (S2c selection behavior)

### Key results to check:
- **Primary delta** (C4_4 vs C4_0): must be >= +0.15 for promotion
- **Family CI lower bound**: must be > 0 for statistical confidence
- **No canonical/abbreviation regression**: no material regression > 0.05
- **Oracle gap capture**: how much of the oracle gap the real pipeline closes

### Next steps (after this notebook):
- If development passes: run qualification split
- If qualification passes: run OOD split
- Gate D decision based on all three splits